# Deep Agent API — SSE Stream Tester
Tests the `/chat` endpoint and renders each SSE event type as it arrives.

In [ ]:
%pip install httpx -q

In [ ]:
import json
import httpx

BASE_URL = "http://localhost:8000"

# Sanity check the server is up
response = httpx.get(f"{BASE_URL}/health")
print(response.json())

In [ ]:
# Emoji labels per event type — purely for readability
EVENT_LABELS = {
    "reasoning": "🧠 [thinking]",
    "tool_call":  "🔧 [tool_call]",
    "tool_result":"📦 [tool_result]",
    "answer":     "💬 [answer]",
    "agent":      "🤖 [subagent]",
    "error":      "❌ [error]",
    "done":       "✅ [done]",
}

In [ ]:
def parse_sse_line(line: str) -> tuple[str, str] | None:
    """
    Parse a raw SSE line into (event, data).
    FastAPI ServerSentEvent emits lines like:
      event: answer
      data: hello world
    Returns None for empty/comment lines.
    """
    line = line.strip()
    if not line or line.startswith(":"):
        return None
    if line.startswith("data:"):
        return ("data", line[len("data:"):].strip())
    if line.startswith("event:"):
        return ("event", line[len("event:"):].strip())
    return None


async def stream_chat(message: str, thread_id: str = "test-1"):
    """
    Stream a chat message to the FastAPI endpoint and print each SSE event.
    SSE events arrive as pairs of lines:
      event: <type>\n
      data: <content>\n
      \n
    We accumulate lines per event block and flush on the blank separator.
    """
    payload = {"message": message, "thread_id": thread_id}
    current_event = "message"  # SSE default if no event: line present
    current_data = None

    async with httpx.AsyncClient(timeout=120) as client:
        async with client.stream("POST", f"{BASE_URL}/chat", json=payload) as response:
            async for raw_line in response.aiter_lines():
                parsed = parse_sse_line(raw_line)

                if parsed is None:
                    # Blank line = end of one SSE event block, flush it
                    if current_data is not None:
                        label = EVENT_LABELS.get(current_event, f"[{current_event}]")
                        print(f"{label} {current_data}")
                        if current_event == "done":
                            break
                    current_event = "message"
                    current_data = None
                    continue

                field, value = parsed
                if field == "event":
                    current_event = value
                elif field == "data":
                    current_data = value

In [ ]:
# Basic test — no tool use, just LLM answer
await stream_chat("What is the capital of France?")

In [ ]:
# Tool use test — should trigger Tavily search
await stream_chat("What is the latest news on Singapore AI policy?", thread_id="test-2")

In [ ]:
# Multi-turn memory test — second message should remember the first
await stream_chat("My name is Titus.", thread_id="test-memory")
print("---")
await stream_chat("What is my name?", thread_id="test-memory")

In [ ]:
# Collect full answer text only — useful for asserting on content
async def get_answer(message: str, thread_id: str = "assert-1") -> str:
    """Returns just the concatenated answer tokens as a string."""
    answer_parts = []
    current_event = "message"
    current_data = None

    async with httpx.AsyncClient(timeout=120) as client:
        async with client.stream("POST", f"{BASE_URL}/chat", json={"message": message, "thread_id": thread_id}) as response:
            async for raw_line in response.aiter_lines():
                parsed = parse_sse_line(raw_line)
                if parsed is None:
                    if current_event == "answer" and current_data:
                        answer_parts.append(current_data)
                    if current_event == "done":
                        break
                    current_event = "message"
                    current_data = None
                    continue
                field, value = parsed
                if field == "event":
                    current_event = value
                elif field == "data":
                    current_data = value

    return "".join(answer_parts)


answer = await get_answer("What is 2 + 2?")
print(f"Answer: {answer}")
assert "4" in answer, f"Expected '4' in answer, got: {answer}"
print("✅ Assertion passed")